In [2]:
from transformers import LlamaTokenizer, LlamaForCausalLM
from peft import PeftModel, PeftConfig
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import random
import torch
import re

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
df_assin_2_treino = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["train"])
df_assin_2_teste = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["test"])
df_assin_2_val = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["validation"])

df_assin_2 = pd.concat([df_assin_2_treino, df_assin_2_teste, df_assin_2_val])

In [5]:
def zero_shot_prompt(premise, hypothesis):
  return f"""
    Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.

    Premissa: {premise}
    Hipótese: {hypothesis}

    Resposta:
    """


In [7]:
tokenizer = LlamaTokenizer.from_pretrained("maritaca-ai/sabia-7b")
model = LlamaForCausalLM.from_pretrained(
    "maritaca-ai/sabia-7b",
    device_map="auto",  # Automatically loads the model in the GPU, if there is one. Requires pip install acelerate
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16   # If your GPU does not support bfloat16, change to torch.float16
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.59G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'sabia_7b'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_sabia_7b.csv')

0 - 2025-12-23 00:29:24
100 - 2025-12-23 00:30:00
200 - 2025-12-23 00:30:35
300 - 2025-12-23 00:31:11
400 - 2025-12-23 00:31:47
500 - 2025-12-23 00:32:22
600 - 2025-12-23 00:32:58
700 - 2025-12-23 00:33:33
800 - 2025-12-23 00:34:09
900 - 2025-12-23 00:34:44
1000 - 2025-12-23 00:35:20
1100 - 2025-12-23 00:35:56
1200 - 2025-12-23 00:36:31
1300 - 2025-12-23 00:37:07
1400 - 2025-12-23 00:37:42
1500 - 2025-12-23 00:38:18
1600 - 2025-12-23 00:38:53
1700 - 2025-12-23 00:39:29
1800 - 2025-12-23 00:40:05
1900 - 2025-12-23 00:40:41
2000 - 2025-12-23 00:41:16
2100 - 2025-12-23 00:41:52
2200 - 2025-12-23 00:42:27
2300 - 2025-12-23 00:43:03
2400 - 2025-12-23 00:43:39
2500 - 2025-12-23 00:44:14
2600 - 2025-12-23 00:44:50
2700 - 2025-12-23 00:45:25
2800 - 2025-12-23 00:46:01
2900 - 2025-12-23 00:46:37
3000 - 2025-12-23 00:47:12
3100 - 2025-12-23 00:47:48
3200 - 2025-12-23 00:48:24
3300 - 2025-12-23 00:48:59
3400 - 2025-12-23 00:49:35
3500 - 2025-12-23 00:50:10
3600 - 2025-12-23 00:50:46
3700 - 2025-1

In [ ]:
df_assin_2.head(200)

In [ ]:
num_records = len(df_assin_2)
random_index = random.randint(0, num_records - 1)
random_index = 2

premissa = df_assin_2.iloc[random_index]['premise']
hipotese = df_assin_2.iloc[random_index]['hypothesis']
prompt = zero_shot_prompt_2(premissa, hipotese)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][prompt_len:]
resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f'Prompt: {prompt}')
print(f'################')
print(f'Resposta: {resp}')

In [ ]:
num_records = len(df_assin_2)
random_index = random.randint(0, num_records - 1)
random_index = 5

premissa = df_assin_2.iloc[random_index]['premise']
hipotese = df_assin_2.iloc[random_index]['hypothesis']
prompt = zero_shot_prompt_4(premissa, hipotese)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][prompt_len:]
resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f'Prompt: {prompt}')
print(f'################')
print(f'Resposta: {resp}')